# 02 — Our own legal BPE tokenizer

**What this notebook does:**

1. Defines the **cleaning function** (decided in `01_data_exploration`): test prompts are pure ASCII with single spaces, so training text must look the same.
2. Builds a **cleaned text sample** from all four sources to learn the vocabulary from.
3. Trains **three tokenizers** (a tokenizer-design experiment, section 7.4):
   - `bpe16k`: 16,000-token vocabulary
   - `bpe32k`: 32,000-token vocabulary
   - `bpe32k_digits`: 32,000 tokens, **every digit is its own token** (idea: our baseline got stuck in loops like `27/87, 27/87` and `2000-2000-…`; numbers split into digits are handled more uniformly)
4. Compares them with GPT-2's tokenizer on the **1,000 real test prompts**: tokens per word, round-trip safety, examples.

**Rules followed:** progress bar on every long step; every result is saved and **loaded instead of recomputed** on re-runs (delete a file to force that step again).


## 0. Setup

Same project-root trick as before: walk up from this notebook's folder until `.gitignore` is found.
Outputs go to `data/tokenizer/` (ignored by Git, because the text sample is large).


In [1]:
import os, json, re, unicodedata, random, time
from pathlib import Path
from itertools import islice
from collections import Counter
import numpy as np
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

TOK_DIR = PROJECT_ROOT / "data" / "tokenizer"
TOK_DIR.mkdir(parents=True, exist_ok=True)
REPO_ID = "Exploration-Lab/CS-781-Capstone"
SEED = 42
print("Project root:", PROJECT_ROOT)
print("Tokenizer folder:", TOK_DIR)

Project root: c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project
Tokenizer folder: c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project\data\tokenizer


c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. The cleaning function

Every rule comes from something we **observed** in `01_data_exploration`:

| Rule                                                                              | Why                                                                     |
| --------------------------------------------------------------------------------- | ----------------------------------------------------------------------- |
| Remove "Signature Not Verified … IST Reason:" stamps                              | PDF signature boxes landed inside SC sentences                          |
| Delete soft hyphens (U+00AD)                                                      | invisible PDF leftovers: `sub­standard` ≠ `substandard` for a tokenizer |
| Map special characters to ASCII (curly quotes → `"`, dashes → `-`, `₹` → `Rs.` …) | the 1,000 test prompts contain **0** non-ASCII characters               |
| Collapse all whitespace to single spaces                                          | the test prompts contain **0** line breaks and **0** double spaces      |

How the ASCII step works:

- `str.translate(table)` replaces characters using a lookup table (we list the common punctuation ourselves).
- `unicodedata.normalize("NFKD", ...)` splits accented letters into letter + accent (`é` → `e` + `´`).
- `.encode("ascii", "ignore")` then keeps only ASCII, so the accent disappears and `é` becomes `e`. Anything still non-ASCII is dropped.

⚠️ **The same function must be used everywhere**: data preparation (`03`) and on prompts before generation. If you ever change it, change it in every notebook.


In [2]:
SIGNATURE_RE = re.compile(r"Signature Not Verified.{0,150}?IST(\s*Reason:)?", flags=re.DOTALL)

ASCII_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",      # curly single quotes
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u201f": '"',      # curly double quotes
    "\u2013": "-", "\u2014": "-", "\u2015": "-", "\u2212": "-",      # en dash, em dash, minus
    "\u2026": "...",                                                 # ellipsis
    "\u00a0": " ", "\u2009": " ", "\u200b": "",                      # non-breaking / thin / zero-width space
    "\u00ad": "",                                                    # soft hyphen (invisible)
    "\u20b9": "Rs.", "\u00a7": "S.", "\u2022": "-", "\u00b7": ".",   # rupee, section sign, bullets
})

def clean_text(text):
    text = SIGNATURE_RE.sub(" ", text)                               # 1. PDF signature stamps
    text = text.translate(ASCII_MAP)                                 # 2. known special characters
    text = unicodedata.normalize("NFKD", text)                       # 3. split accents from letters
    text = text.encode("ascii", "ignore").decode("ascii")            #    ...and keep ASCII only
    return " ".join(text.split())                                    # 4. all whitespace -> one space

# Quick test on the kinds of problems we found
demo = ("dismissed the said criminal petition Signature Not Verified Digitally signed by ARJUN BISHT "
        "Date: 2020.01.31 16:39:59 IST Reason: Reasons for the sub\u00adstandard \u201cmaterial\u201d \u2013 "
        "Rs \u20b9500 under \u00a7 420.\n\nNext  paragraph caf\u00e9.")
print("BEFORE:", repr(demo))
print("AFTER :", repr(clean_text(demo)))

BEFORE: 'dismissed the said criminal petition Signature Not Verified Digitally signed by ARJUN BISHT Date: 2020.01.31 16:39:59 IST Reason: Reasons for the sub\xadstandard “material” – Rs ₹500 under § 420.\n\nNext  paragraph café.'
AFTER : 'dismissed the said criminal petition Reasons for the substandard "material" - Rs Rs.500 under S. 420. Next paragraph cafe.'


## 2. Build the tokenizer training sample

A tokenizer doesn't need the whole corpus; ~150 MB of text is plenty to learn which character sequences are frequent.
We take text from **all four sources** so the vocabulary covers every style, weighted toward the judgment style of the test prompts:

| Source        | Characters | How we read it                                                                                                   |
| ------------- | ---------- | ---------------------------------------------------------------------------------------------------------------- |
| Supreme Court | 60 M       | the **downloaded** file, streamed locally with `ijson`, keeping every 15th document (spread over the whole file) |
| Tribunals     | 45 M       | streamed from Hugging Face (first documents)                                                                     |
| Acts          | 20 M       | streamed from Hugging Face                                                                                       |
| High Courts   | 25 M       | streamed, **only documents with ≥ 200 words** (skip the short procedural orders)                                 |

Each cleaned document becomes **one line** of `corpus_sample.txt` (possible because cleaning removed all line breaks).
The file is saved; if it exists, this cell skips the whole step.

Limitation to note in the report: for the three streamed files we read the **start** of each file, which may not represent the whole file.


In [3]:
import ijson
from huggingface_hub import HfFileSystem, hf_hub_download

SAMPLE_FILE = TOK_DIR / "corpus_sample.txt"
BUDGET = {                          # characters of cleaned text per source
    "train_supreme_court.json": 60_000_000,
    "train_tribunals.json":     45_000_000,
    "train_acts.json":          20_000_000,
    "train_high_courts.json":   25_000_000,
}
MIN_WORDS = {"train_high_courts.json": 200}   # skip short HC orders
SC_EVERY = 15                                  # SC: keep every 15th document

fs = HfFileSystem()

def records_of(name):
    # Yield records one at a time. SC comes from the local download, the rest are streamed.
    if name == "train_supreme_court.json":
        path = hf_hub_download(REPO_ID, f"Phase_1_Pretraining/Train/{name}", repo_type="dataset")
        f = open(path, "rb")
        items = islice(ijson.items(f, "item"), 0, None, SC_EVERY)
    else:
        f = fs.open(f"datasets/{REPO_ID}/Phase_1_Pretraining/Train/{name}", "rb", block_size=2**20)
        items = ijson.items(f, "item")
    try:
        for rec in items:
            yield rec
    finally:
        f.close()

if SAMPLE_FILE.exists():
    print(f"[cache] sample exists ({SAMPLE_FILE.stat().st_size/1e6:.0f} MB) - skipping")
else:
    tmp = SAMPLE_FILE.with_suffix(".tmp")      # write to a temp file, rename at the end -> no half-finished sample
    with open(tmp, "w", encoding="ascii") as out:
        for name, budget in BUDGET.items():
            n_chars = n_docs = 0
            bar = tqdm(total=budget, desc=name, unit="char", unit_scale=True)
            for rec in records_of(name):
                if rec.get("word_count", 10**9) < MIN_WORDS.get(name, 0):
                    continue
                text = clean_text(rec.get("text") or "")
                if len(text.split()) < 50:     # drop empty / near-empty documents
                    continue
                out.write(text + "\n")
                n_chars += len(text); n_docs += 1
                bar.update(len(text))
                if n_chars >= budget:
                    break
            bar.close()
            print(f"  {name}: {n_docs:,} docs, {n_chars/1e6:.1f} M chars")
    tmp.rename(SAMPLE_FILE)
    print("Saved", SAMPLE_FILE)

train_supreme_court.json: 60.0Mchar [00:04, 13.8Mchar/s]                         


  train_supreme_court.json: 2,639 docs, 60.0 M chars


train_tribunals.json: 45.1Mchar [00:50, 899kchar/s]                            


  train_tribunals.json: 1,478 docs, 45.1 M chars


train_acts.json: 20.6Mchar [00:20, 990kchar/s]                          


  train_acts.json: 504 docs, 20.6 M chars


train_high_courts.json: 25.0Mchar [00:31, 796kchar/s]                          

  train_high_courts.json: 7,384 docs, 25.0 M chars
Saved c:\Users\Atharva\Desktop\Assignments\LLM\Capstone Project\data\tokenizer\corpus_sample.txt


## 3. Train the three BPE tokenizers

**BPE, reminder:** start from single characters (bytes), count every adjacent pair in the sample, merge the most frequent pair into a new token, repeat until the vocabulary reaches `vocab_size`.

Settings:

- **Byte-level** (`ByteLevel` pre-tokenizer + decoder): the base alphabet is all 256 byte values, so _any_ text can be encoded, and `decode(encode(x))` gives back exactly `x`. A space becomes `Ġ`, like GPT-2.
- **Special tokens:** `<|endoftext|>` (id 0, document boundary / EOS) and `<pad>` (id 1). Separate pad and EOS avoid the starter's "pad = EOS" confusion.
- `min_frequency=2`: a pair must appear at least twice to be merged.
- **Digits variant:** the `Digits(individual_digits=True)` pre-tokenizer splits every number into single digits _before_ BPE, so `1996` is always `1`,`9`,`9`,`6`. LLaMA does this. Costs more tokens per number, but the model sees numbers consistently.

Each tokenizer is saved as one `.json` file; existing ones are loaded instead of retrained. The `tokenizers` library draws its own progress bar.


In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SPECIAL_TOKENS = ["<|endoftext|>", "<pad>"]
VARIANTS = {
    "bpe16k":        {"vocab_size": 16_000, "digits": False},
    "bpe32k":        {"vocab_size": 32_000, "digits": False},
    "bpe32k_digits": {"vocab_size": 32_000, "digits": True},
}

def train_tokenizer(vocab_size, digits):
    tok = Tokenizer(models.BPE())
    byte_level = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.pre_tokenizer = (pre_tokenizers.Sequence([pre_tokenizers.Digits(individual_digits=True), byte_level])
                         if digits else byte_level)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # all 256 bytes -> nothing is ever "unknown"
        show_progress=True,
    )
    tok.train([str(SAMPLE_FILE)], trainer)
    return tok

tokenizers_ = {}
for name, cfg in VARIANTS.items():
    path = TOK_DIR / f"{name}.json"
    if path.exists():
        tokenizers_[name] = Tokenizer.from_file(str(path))
        print(f"[cache] loaded {name}")
    else:
        t0 = time.time()
        tokenizers_[name] = train_tokenizer(cfg["vocab_size"], cfg["digits"])
        tokenizers_[name].save(str(path))
        print(f"trained {name} in {time.time()-t0:.0f}s -> {path.name}")
    print(f"  vocab size: {tokenizers_[name].get_vocab_size():,}  | EOS id: {tokenizers_[name].token_to_id('<|endoftext|>')}")

trained bpe16k in 19s -> bpe16k.json
  vocab size: 16,000  | EOS id: 0
trained bpe32k in 7s -> bpe32k.json
  vocab size: 32,000  | EOS id: 0
trained bpe32k_digits in 8s -> bpe32k_digits.json
  vocab size: 32,000  | EOS id: 0


## 4. Compare on the real test prompts

For each tokenizer (ours + GPT-2's) we measure on all **1,000 test prompts** (512 words each):

- **tokens per word** (a.k.a. _fertility_): lower = the same text needs fewer tokens → more text per training block, faster training, and a 512-word prompt uses less of the context window.
- **tokens per prompt** (mean and max): decides the context length we need (prompt + ~128 words of output must fit).
- **round-trip check**: `decode(encode(text)) == text` for every prompt (a tokenizer that changes text is broken).
- **embedding parameters** = vocab size × width (560 like the baseline), to show the parameter saving.

Measuring token counts on the test _inputs_ is fine: we only count, we never train on them.
Results are cached in `tokenizer_comparison.json`.


In [5]:
from transformers import AutoTokenizer

test_path = hf_hub_download(REPO_ID, "Phase_1_Pretraining/Test/test.json", repo_type="dataset")
test_prompts = [r["text"] for r in json.load(open(test_path, encoding="utf-8")).values()]
n_words = sum(len(p.split()) for p in test_prompts)

gpt2 = AutoTokenizer.from_pretrained("openai-community/gpt2")
encoders = {"gpt2 (50k)": lambda s: gpt2(s, add_special_tokens=False)["input_ids"]}
decoders_ = {"gpt2 (50k)": lambda ids: gpt2.decode(ids)}
for name, tok in tokenizers_.items():
    encoders[name] = lambda s, t=tok: t.encode(s).ids          # t=tok freezes the current tokenizer in the lambda
    decoders_[name] = lambda ids, t=tok: t.decode(ids)
vocab = {"gpt2 (50k)": len(gpt2), **{n: t.get_vocab_size() for n, t in tokenizers_.items()}}

COMPARE_FILE = TOK_DIR / "tokenizer_comparison.json"
if COMPARE_FILE.exists():
    results = json.load(open(COMPARE_FILE))
    print("[cache] loaded comparison")
else:
    results = {}
    for name, enc in encoders.items():
        lens, roundtrip_ok = [], 0
        for p in tqdm(test_prompts, desc=name):
            ids = enc(p)
            lens.append(len(ids))
            roundtrip_ok += decoders_[name](ids) == p
        results[name] = {"vocab": vocab[name], "tokens_per_word": sum(lens) / n_words,
                         "mean_tokens": float(np.mean(lens)), "max_tokens": int(max(lens)),
                         "roundtrip_ok": roundtrip_ok, "embedding_params_M": vocab[name] * 560 / 1e6}
    json.dump(results, open(COMPARE_FILE, "w"), indent=2)

print(f"{'tokenizer':16s} {'vocab':>7s} {'tok/word':>9s} {'mean tok':>9s} {'max tok':>8s} {'round-trip':>11s} {'emb params':>11s}")
for name, r in results.items():
    print(f"{name:16s} {r['vocab']:7,d} {r['tokens_per_word']:9.3f} {r['mean_tokens']:9.0f} {r['max_tokens']:8d} "
          f"{r['roundtrip_ok']:>5d}/1000 {r['embedding_params_M']:9.1f} M")

bpe32k_digits: 100%|██████████| 1000/1000 [00:00<00:00, 1057.92it/s]

tokenizer          vocab  tok/word  mean tok  max tok  round-trip  emb params
gpt2 (50k)        50,257     1.240       635      967  1000/1000      28.1 M
bpe16k            16,000     1.238       634     1086  1000/1000       9.0 M
bpe32k            32,000     1.198       613      982  1000/1000      17.9 M
bpe32k_digits     32,000     1.280       655     1090  1000/1000      17.9 M


## 5. See how each tokenizer splits legal text

Numbers, citations and legal terms are where tokenizers differ most. `|` separates tokens; `Ġ` means "a space came before this".


In [6]:
examples = [
    "The appellant filed a writ petition under Article 226 of the Constitution.",
    "(Vide Hari Singh & Ors. Vs. State of U.P., AIR 1984 SC 1020).",
    "Section 420 read with 120B IPC; Criminal Appeal No. 125 of 2020 dated 07.09.2018.",
    "The respondent-landlord sought eviction under the Rent Control Act, 1958.",
]
for ex in examples:
    print("TEXT:", ex)
    print(f"  {'gpt2 (50k)':14s} ({len(gpt2.tokenize(ex)):2d}) " + "|".join(gpt2.tokenize(ex)))
    for name, tok in tokenizers_.items():
        toks = tok.encode(ex).tokens
        print(f"  {name:14s} ({len(toks):2d}) " + "|".join(toks))
    print()

TEXT: The appellant filed a writ petition under Article 226 of the Constitution.
  gpt2 (50k)     (13) The|Ġappellant|Ġfiled|Ġa|Ġwrit|Ġpetition|Ġunder|ĠArticle|Ġ226|Ġof|Ġthe|ĠConstitution|.
  bpe16k         (13) The|Ġappellant|Ġfiled|Ġa|Ġwrit|Ġpetition|Ġunder|ĠArticle|Ġ226|Ġof|Ġthe|ĠConstitution|.
  bpe32k         (13) The|Ġappellant|Ġfiled|Ġa|Ġwrit|Ġpetition|Ġunder|ĠArticle|Ġ226|Ġof|Ġthe|ĠConstitution|.
  bpe32k_digits  (16) The|Ġappellant|Ġfiled|Ġa|Ġwrit|Ġpetition|Ġunder|ĠArticle|Ġ|2|2|6|Ġof|Ġthe|ĠConstitution|.

TEXT: (Vide Hari Singh & Ors. Vs. State of U.P., AIR 1984 SC 1020).
  gpt2 (50k)     (24) (|V|ide|ĠHar|i|ĠSingh|Ġ&|ĠOr|s|.|ĠVs|.|ĠState|Ġof|ĠU|.|P|.,|ĠAIR|Ġ1984|ĠSC|Ġ10|20|).
  bpe16k         (21) (|Vide|ĠHari|ĠSingh|Ġ&|ĠOrs|.|ĠVs|.|ĠState|Ġof|ĠU|.|P|.,|ĠAIR|Ġ1984|ĠSC|Ġ10|20|).
  bpe32k         (21) (|Vide|ĠHari|ĠSingh|Ġ&|ĠOrs|.|ĠVs|.|ĠState|Ġof|ĠU|.|P|.,|ĠAIR|Ġ1984|ĠSC|Ġ10|20|).
  bpe32k_digits  (28) (|Vide|ĠHari|ĠSingh|Ġ&|ĠOrs|.|ĠVs|.|ĠState|Ġof|ĠU|.|P|.,|ĠAIR|Ġ|1|9|8|4|ĠS

## Viva check

1. Why did we clean the text _before_ training the tokenizer, and why must the same cleaning be applied to test prompts?
2. What does "byte-level" guarantee that a character-level BPE doesn't?
3. Why is fewer tokens per word good for training speed and for the context window?
4. How does vocabulary size change the model's parameter count? (hint: embedding matrix = vocab × width)
5. Why might splitting numbers into single digits help a model with dates and citations — and what does it cost?
6. Why can't you compare validation perplexity between models with different tokenizers? What do you compare instead?
